# ABS Earnings by Educational Attainment (6337.0)

Median weekly earnings by highest non-school qualification, from *Characteristics of Employment*
(cat. 6337.0, formerly 6333.0), August 2025 release, Table 6 (Data 6 sheet).

Filter: Australia, all persons, total employees, median weekly earnings, all industries/occupations.
Note: this is **all employees** (full-time + part-time). Median full-time-only is not directly
published in this table; the no-qualification group has more part-time workers, so its median
is depressed relative to a full-time cut.

## Set-up

In [1]:
from pathlib import Path

import pandas as pd
from mgplot import bar_plot_finalise, line_plot_finalise, set_chart_dir, clear_chart_dir

In [2]:
DATA_PATH = Path('./ABS-Data/6337.0/Table_6.xlsx')
CHART_DIR = './CHARTS/6337.0 - Earnings by Education/'
Path(CHART_DIR).mkdir(parents=True, exist_ok=True)
set_chart_dir(CHART_DIR)
clear_chart_dir()

SHOW = False
SURVEY_MONTH = 'Aug-25'

DIM_COLS = ['Survey month', 'Parameter', 'State', 'Leave', 'Sex', 'Classification', 'Category']
QUAL_COLS = [
    'Postgraduate Degree', 'Graduate Diploma or Certificate', 'Bachelor Degree',
    'Bachelor Degree or Higher Total', 'Advanced Diploma or Diploma',
    'Certificate III or IV', 'Other non-school qualification',
    'With non-school qualification Total', 'Without non-school qualification', 'Total',
]

BAR_QUALS = [
    ('Postgraduate Degree', 'Postgraduate'),
    ('Bachelor Degree', 'Bachelor'),
    ('Advanced Diploma or Diploma', 'Diploma'),
    ('Certificate III or IV', 'Certificate III/IV'),
    ('Without non-school qualification', 'No post-school qual.'),
]

## Load and shape

In [3]:
def load_data() -> pd.DataFrame:
    """Load Data 6 sheet with proper column names.

    The sheet has 4 banner rows above the data; row 7 (0-indexed) is the column header,
    but it mixes dimension labels with the qualification group hierarchy from rows 4-6.
    We skip 8 rows and name the columns directly.
    """
    df = pd.read_excel(DATA_PATH, sheet_name='Data 6', header=None, skiprows=8)
    df.columns = DIM_COLS + [c for q in QUAL_COLS for c in (q, q + ' RSE')]
    return df


def get_national_medians(df: pd.DataFrame, month: str) -> pd.Series:
    """Pick the single 'Australia / Persons / Total / Total employees / Median weekly earnings'
    row for the chosen month and return median weekly earnings by qualification."""
    row = df[
        (df['State'] == 'Australia')
        & (df['Sex'] == 'Persons')
        & (df['Parameter'] == 'Median weekly earnings')
        & (df['Survey month'] == month)
        & (df['Classification'] == 'Total')
        & (df['Leave'] == 'Total employees')
    ]
    if len(row) != 1:
        raise ValueError(f'Expected 1 row, got {len(row)}')
    return row.iloc[0][QUAL_COLS].astype(float)


df = load_data()
medians = get_national_medians(df, SURVEY_MONTH)
print(medians.round(0))

Postgraduate Degree                    2000.0
Graduate Diploma or Certificate        1900.0
Bachelor Degree                        1700.0
Bachelor Degree or Higher Total        1766.0
Advanced Diploma or Diploma            1462.0
Certificate III or IV                  1400.0
Other non-school qualification         1300.0
With non-school qualification Total    1600.0
Without non-school qualification       1000.0
Total                                  1425.0
Name: 88440, dtype: float64


## Plot

In [4]:
def plot_bar(medians: pd.Series, month: str) -> None:
    """Bar chart of median weekly earnings for selected qualification levels."""
    selected = pd.Series(
        {label: medians[qual] for qual, label in BAR_QUALS},
        name='Median weekly earnings ($)',
    )
    bar_plot_finalise(
        selected.to_frame(),
        title=f'Median Weekly Earnings by Highest Qualification: {month}',
        ylabel='$ per week',
        annotate=True,
        rounding=0,
        color=['steelblue'],
        legend=False,
        rfooter='ABS 6337.0 T6',
        lfooter=(
            'Australia. All employees (full-time + part-time). '
            'Median weekly earnings in main job. '
        ),
        show=SHOW,
    )


plot_bar(medians, SURVEY_MONTH)

## Growth over time

Median weekly earnings indexed to Aug-14 = 100 for each qualification level. Shows the no-qualification group growing slowest over 2014-25.

In [5]:
BASE_MONTH = "Aug-14"


def get_yearly_medians(df: pd.DataFrame) -> pd.DataFrame:
    """All survey-month medians for Australia, Persons, Total, Total employees."""
    sub = df[
        (df["State"] == "Australia")
        & (df["Sex"] == "Persons")
        & (df["Parameter"] == "Median weekly earnings")
        & (df["Classification"] == "Total")
        & (df["Leave"] == "Total employees")
    ].copy()
    sub.index = pd.PeriodIndex(
        sub["Survey month"].str.replace("Aug-", "20") + "Q3", freq="Q"
    )
    return sub[QUAL_COLS].astype(float).sort_index()


def plot_growth(df: pd.DataFrame, base_month: str) -> None:
    """Index each qualification series to the base month = 100 and plot."""
    yearly = get_yearly_medians(df)
    base_period = pd.Period(base_month.replace("Aug-", "20") + "Q3", freq="Q")
    indexed = yearly.div(yearly.loc[base_period]) * 100

    selected_cols = [q for q, _ in BAR_QUALS]
    selected_labels = [label for _, label in BAR_QUALS]
    indexed = indexed[selected_cols]
    indexed.columns = selected_labels

    line_plot_finalise(
        indexed,
        title=f"Median Weekly Earnings by Qualification: Indexed to {base_month}",
        ylabel=f"Index ({base_month} = 100)",
        width=2,
        annotate=True,
        rounding=0,
        rfooter="ABS 6337.0 T6",
        lfooter="Australia. All employees. Median weekly earnings in main job. ",
        show=SHOW,
    )


plot_growth(df, BASE_MONTH)


## Nominal earnings over time

Same series as above but in nominal dollars per week, not indexed.

In [6]:
def plot_nominal(df: pd.DataFrame) -> None:
    """Median weekly earnings ($/week) by qualification across all survey months."""
    yearly = get_yearly_medians(df)
    selected_cols = [q for q, _ in BAR_QUALS]
    selected_labels = [label for _, label in BAR_QUALS]
    series = yearly[selected_cols]
    series.columns = selected_labels

    line_plot_finalise(
        series,
        title="Median Weekly Earnings by Qualification",
        ylabel="$ per week",
        width=2,
        annotate=True,
        rounding=0,
        rfooter="ABS 6337.0 T6",
        lfooter="Australia. All employees. Median weekly earnings in main job. ",
        show=SHOW,
    )


plot_nominal(df)


## Watermark

In [7]:
%load_ext watermark
%watermark -u -t -d --iversions

Last updated: 2026-05-11 19:18:04

mgplot : 0.2.23
pandas : 3.0.2
pathlib: 1.0.1

